In [32]:
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from torch_geometric.utils import k_hop_subgraph, to_undirected
import scipy.sparse as sp

from retail_data_prep import preprocess_events, apply_activity_threshold
from data_splitting import reindex_nodes
from modules.retail_data_prep import preprocess_events, apply_activity_threshold
from modules.data_splitting import reindex_nodes
from modules.subgraph_dataclass import LinkSubgraphDataset
from modules.model_specifications import  BaselineGCNSubgraphEncoder, GATOnlySubgraphEncoder, PGADRLSubgraphEncoder





def ndcg_k(pred_items, true_items, k):
    """Compute NDCG@k."""
    dcg = 0.0
    for idx, item in enumerate(pred_items[:k], start=1):
        if item in true_items:
            dcg += 1.0 / np.log2(idx + 1)

    ideal_hits = min(k, len(true_items))
    idcg = sum(1.0 / np.log2(i + 1) for i in range(1, ideal_hits + 1))
    return dcg / idcg if idcg > 0 else 0.0


def recall_k(pred_items, true_items, k):
    """Compute Recall@k."""
    return len(set(pred_items[:k]).intersection(true_items)) / max(1, len(true_items))

def sample_negatives_for_user(user_id, num_items, user_pos_items, n=100):
    """
    Samples 100 random items user has never interacted with.
    """
    excluded = set(user_pos_items)
    excluded.add(-1)  # guard
    candidates = []
    while len(candidates) < n:
        i = np.random.randint(0, num_items)
        if i not in excluded:
            candidates.append(i)
    return candidates

def build_subgraph_features(u, i, full_edge_index, hops=1):
    """
    Builds enclosing subgraph for (u, i) using k-hop neighborhood.
    """
    # Extract k-hop subgraph nodes and edges
    nodes, sub_edge_index, _, _ = k_hop_subgraph(
        node_idx=[u, i],      # center nodes
        num_hops=hops,
        edge_index=full_edge_index,
        relabel_nodes=True
    )
    sub_edge_index = to_undirected(sub_edge_index)

    # DRNL node labels
    x = drnl_labeling(
        sub_nodes=nodes,
        u_idx=0,    # After relabeling center nodes move to 0 and 1
        v_idx=1
    ).unsqueeze(-1)

    # Batch = all zeros since this is one graph
    batch = torch.zeros(x.size(0), dtype=torch.long)
    return x.float(), sub_edge_index.long(), batch



def gnn_score(model, u, i, full_edge_index, hops):
    """
    Runs the GNN on a single (u,i) edge.
    """
    x_sub, ei_sub, batch = build_subgraph_features(u, i, full_edge_index, hops=hops)
    with torch.no_grad():
        logit = model(x_sub.to(device), ei_sub.to(device), batch.to(device))
    return float(logit.item())


def evaluate_ranking(models, events, train_mask, eval_mask, hops=1, k=10):
    """
    Compute Recall@K and NDCG@K for models
    """

    eval_df = events[eval_mask][["user_idx", "item_idx"]]
    train_df = events[train_mask][["user_idx", "item_idx"]]

    num_users = events["user_idx"].max() + 1
    num_items = events["item_idx"].max() + 1

    # Build full user->items interactions for positive sets
    user_pos_items = (
        eval_df.groupby("user_idx")["item_idx"]
        .apply(set)
        .to_dict()
    )

    # Filter warm users >= 10 interactions in train
    warm_user_counts = train_df.groupby("user_idx").size()
    warm_users = set(warm_user_counts[warm_user_counts >= 10].index)
    eval_users = [u for u in user_pos_items if u in warm_users]

    results = {name: {"recall": [], "ndcg": []} for name in models}

    # Make bipartite UI for ALS
    rows = train_df["user_idx"].values
    cols = train_df["item_idx"].values
    data_vals = np.ones_like(rows)
    user_item_matrix = sp.csr_matrix(
        (data_vals, (rows, cols)),
        shape=(num_users, num_items)
    )

    als_model = models.get("ALS", None)

    # For GNNs, prepare edge index
    full_edge_index = torch.tensor(
        [train_df["user_idx"].tolist(), train_df["item_idx"].tolist()],
        dtype=torch.long
    ).to(device)

    for u in tqdm(eval_users, desc="Evaluating users"):

        true_items = user_pos_items[u]

        # Sample 100 random negatives
        neg_items = sample_negatives_for_user(
            u, num_items, user_pos_items[u], n=100
        )

        # Candidate set = true positives + negatives
        candidates = list(true_items) + neg_items

        for name, model in models.items():

            scores = []

            for item in candidates:

                if name == "ALS":
                    # ALS recommends by factor dot-product
                    user_f = als_model.user_factors[u]
                    item_f = als_model.item_factors[item]
                    s = np.dot(user_f, item_f)

                elif name == "LogisticRegression":
                    # Logistic regression should take (user,item) features
                    # Placeholder: must plug in your actual LR feature builder
                    feat = np.array([
                        warm_user_counts.get(u, 0),
                        warm_user_counts.get(item, 0)
                    ])
                    s = model.predict_proba([feat])[0][1]

                else:
                    # GNN-based scoring
                    model.eval()
                    s = gnn_score(model, u, item, full_edge_index, hops)

                scores.append(s)

            # Rank candidates
            ranked = [c for _, c in sorted(zip(scores, candidates), reverse=True)]

            # Compute metrics
            rec = recall_k(ranked, true_items, k)
            ndcg = ndcg_k(ranked, true_items, k)

            results[name]["recall"].append(rec)
            results[name]["ndcg"].append(ndcg)

    # Average metrics
    for name in results:
        results[name]["recall"] = np.mean(results[name]["recall"])
        results[name]["ndcg"]   = np.mean(results[name]["ndcg"])

    return results


    events = pd.read_parquet("data/processed_events.parquet")

    # Temporal split definition
    train_mask = events["month"].isin([5, 6])
    val_mask   = events["month"] == 7
    test_mask  = events["month"] == 8


    als_model = implicit.als.AlternatingLeastSquares(factors=64)
    als_model = als_model.load("models/als_model.npz")

    # GNN models (load your best checkpoints)
    gcn   = BaselineGCNSubgraphEncoder(in_channels=1).to(device)
    gcn.load_state_dict(torch.load("models/gcn_best.pt"))

    gat   = GATOnlySubgraphEncoder(in_channels=1).to(device)
    gat.load_state_dict(torch.load("models/gat_best.pt"))

    hybrid = PGADRLSubgraphEncoder(in_channels=1).to(device)
    hybrid.load_state_dict(torch.load("models/hybrid_best.pt"))

    models = {
        "ALS": als_model,
        "GCN": gcn,
        "GAT": gat,
        "Hybrid": hybrid,
        # "LogisticRegression": lr_model   # plug in if you add it
    }


    results = evaluate_ranking(
        models=models,
        events=events,
        train_mask=train_mask,
        eval_mask=test_mask,
        hops=1,
        k=10
    )

    print("\n Ranking Results (Test)")
    for name, metrics in results.items():
        print(f"{name:12s} | Recall@10={metrics['recall']:.4f} | NDCG@10={metrics['ndcg']:.4f}")


FileNotFoundError: [Errno 2] No such file or directory: 'data/processed_events.parquet'